In [1]:
import re
import json
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional
import random
from google.colab import drive, ai

# Directory paths
GROUND_TRUTH ="/content/drive/MyDrive/AI6129/ground_truth"
OUTPUT_FOLDER = "/content/drive/MyDrive/AI6129/accession/validation_splits"
XML_FOLDER = "/content/drive/MyDrive/AI6129/xml"
FORMAT_RULES_FILE = "/content/drive/MyDrive/AI6129/accession_formats.md"

# Stratification bins
STRATA_BINS = {
    "zero": (0, 0),       # 0 accessions
    "low": (1, 2),        # 1-2 accessions
    "medium": (3, 5),     # 3-5 accessions
    "high": (6, float('inf'))  # 6+ accessions
}

# Split ratios (must sum to 1.0)
SPLIT_RATIOS = {
    "development": 0.20,  # 70 documents (rapid iteration)
    "validation": 0.51,   # 180 documents (model comparison)
    "test": 0.29          # 102 documents (held-out final evaluation)
}

PINNED_DOCUMENTS = {
    "development": [
        "PMC9387262",  # 556 accessions - extreme outlier, use for tuning
    ],
    "validation": [],
    "test": []
}

RANDOM_SEED = 42  # For reproducibility

In [2]:
# Mount Google Drive
print("\n1. Mounting Google Drive...")
try:
    drive.mount('/content/drive', force_remount=False)
    print("   Google Drive mounted")
except Exception as e:
    print(f"  Error mounting Drive: {e}")




1. Mounting Google Drive...
Mounted at /content/drive
   Google Drive mounted


In [3]:
def extract_pmcid_from_filename(filename: str) ->Optional[str]:
    """
    Extract PMC ID from JSON filename.

    Expected patterns:
    - PMC1234567.json
    - PMC1234567_metadata.json
    - pmc1234567.json (case-insensitive)

    Returns:
        PMC ID string (e.g., "PMC1234567") or None if not found
    """

    # Pattern: PMC followed by digits
    pattern = r'(PMC\d+)'
    match = re.search(pattern, filename, re.IGNORECASE)

    if match:
        return match.group(1).upper()

    return None

# HELPER: PARSE ACCESSIONS FROM GROUND TRUTH

In [4]:

def parse_ground_truth_accessions(data: dict) -> List[str]:
    """
    Extract accession numbers from ground truth JSON.

    Handles formats:
    - "merge_accession_number": list

    Returns:
        List of unique accession strings
    """
    accessions = []

    #Try list format first
    acc_list = data.get("merge_accession_number", [])

    if acc_list and isinstance(acc_list, list):
        accessions.extend([item.strip() for item in acc_list if item and isinstance(item, str)])

    elif acc_list and isinstance(acc_list, str):
        #split by common delimiter
        raw_items = re.split(r'[,;\s\n]+', acc_list)
        accessions.extend([item.strip() for item in raw_items if item.strip()])

    # Strip [xxx] suffix from each accession  # changed
    accessions = [re.sub(r'\[.*?\]$', '', acc) for acc in accessions]

    #remove duplicates but keep the order
    seen = set()
    unique_accessions = []
    for acc in accessions:
        acc_normalised = acc.strip()
        if acc_normalised and acc_normalised not in seen:
            seen.add(acc_normalised)
            unique_accessions.append(acc_normalised)

    return unique_accessions


In [5]:
def get_accession_stratum(count: int) -> str:
    """
    Determine which stratum a document belongs to based on accession count.

    Returns:
        Stratum name: "zero", "low", "medium", or "high"
    """

    for stratum_name, (min_val, max_val) in STRATA_BINS.items():
      if min_val <= count <= max_val:
        return stratum_name
    return "high" #default fallback


TASK A: ANALYSE GROUND TRUTH DISTRIBUTION

In [6]:
def load_all_ground_truth(folder_path: str) -> Dict[str, dict]:
    """
    Load all JSON ground truth files from folder.

    Returns:
        Dictionary mapping PMCID to document data
    """

    ground_truth_dict = {}
    errors = []

    json_files = list(Path(folder_path).glob("*.json"))
    print(f"Found {len(json_files)} JSON files in folder")

    for gt_file in json_files:
        try:
            with open(gt_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            #Extract PMCID
            pmcid = extract_pmcid_from_filename(gt_file.name)

            if not pmcid:
                errors.append(f"Could not extract PMCID from: {gt_file.name}")
                continue

            #parse accession
            accessions = parse_ground_truth_accessions(data)
            accession_count = len(accessions)
            stratum = get_accession_stratum(accession_count)

            ground_truth_dict[pmcid] = {
                "pmcid": pmcid,
                "filename": gt_file.name,
                "filepath": str(gt_file),
                "accessions": accessions,
                "accession_count": accession_count,
                "stratum": stratum,
                "raw_data": data
            }

        except json.JSONDecodeError as e:
            errors.append(f"JSON parse error in {gt_file.name}: {e}")
        except Exception as e:
            errors.append(f"error loading {gt_file.name}: {e}")

    if errors:
        print(f"\nWarning: {len(errors)} files had errors:")
        for err in errors[:10]:  # Show first 10
            print(f"  - {err}")
        if len(errors) > 10:
            print(f"  ... and {len(errors) - 10} more")

    return ground_truth_dict



In [7]:
def analyse_ground_truth_distribution(ground_truth_dict: Dict[str, dict]) -> dict:
    """
    Analyse the distribution of accession counts across all documents.

    Returns:
        Statistics dictionary with distribution data
    """
    statistics = {
        "total_documents": len(ground_truth_dict),
        "accession_count_distribution": [],
        "strata_counts": Counter(),
        "strata_documents": {stratum: [] for stratum in STRATA_BINS.keys()},
        "empty_ground_truth_count": 0,
        "max_accessions": 0,
        "min_accessions": float('inf'),
        "total_accessions": 0,
    }

    for pmcid, doc_data in ground_truth_dict.items():
      count = doc_data["accession_count"]
      stratum = doc_data["stratum"]

      statistics["accession_count_distribution"].append(count)
      statistics["strata_counts"][stratum] += 1
      statistics["strata_documents"][stratum].append(pmcid)
      statistics["total_accessions"] += count

      if count == 0:
          statistics["empty_ground_truth_count"] += 1

      if count > statistics["max_accessions"]:
          statistics["max_accessions"] = count

      if count < statistics["min_accessions"]:
          statistics["min_accessions"] = count

      #calculate mean

      if statistics["total_documents"] > 0 :
          statistics["mean_accessions"] = (
              statistics["total_accessions"] / statistics["total_documents"]
          )
      else:
          statistics["mean_accessions"] = 0

      #calculate count distribution

      count_freq = Counter(statistics["accession_count_distribution"])
      statistics["count_frequency"] = dict(sorted(count_freq.items()))

    return statistics

In [8]:
def print_distribution_report(statistics: dict) -> None:
    """
    Print a formatted report of the accession distribution.
    """
    print("\n" + "=" * 70)
    print("GROUND TRUTH ACCESSION DISTRIBUTION ANALYSIS")
    print("=" * 70)

    print(f"\nTotal Documents: {statistics['total_documents']}")
    print(f"Total Accessions: {statistics['total_accessions']}")
    print(f"Mean Accessions per Document: {statistics['mean_accessions']:.2f}")
    print(f"Min Accessions: {statistics['min_accessions']}")
    print(f"Max Accessions: {statistics['max_accessions']}")
    print(f"Documents with Zero Accessions: {statistics['empty_ground_truth_count']}")

    print("\n" + "-" * 70)
    print("STRATIFICATION SUMMARY")
    print("-" * 70)
    print(f"{'Stratum':<12} {'Range':<12} {'Count':<10} {'Percentage':<12}")
    print("-" * 70)

    total = statistics["total_documents"]

    for stratum_name, (min_val, max_val) in STRATA_BINS.items():
        count = statistics["strata_counts"].get(stratum_name, 0)
        pct = (count / total * 100) if total > 0 else 0
        range_str = f"{min_val}" if min_val == max_val else f"{min_val}-{int(max_val) if max_val != float('inf') else '+'}"
        print(f"{stratum_name:<12} {range_str:<12} {count:<10} {pct:.1f}%")

    print("\n" + "-" * 70)
    print("DETAILED COUNT FREQUENCY")
    print("-" * 70)
    print(f"{'Count':<10} {'Documents':<15} {'Percentage':<12}")
    print("-" * 70)

    for count, freq in sorted(statistics["count_frequency"].items())[:20]:  # Show first 20
        pct = (freq / total * 100) if total > 0 else 0
        print(f"{count:<10} {freq:<15} {pct:.1f}%")

    if len(statistics["count_frequency"]) > 20:
        print(f"... and {len(statistics['count_frequency']) - 20} more unique counts")

# TASK C: CREATE STRATIFIED SAMPLE

In [9]:
def create_stratified_three_set_split(
    ground_truth_dict: Dict[str, dict],
    split_ratios: Dict[str, float] = None,
    random_seed: int = RANDOM_SEED,
    pinned_documents: Dict[str, List[str]] = None
) -> Dict[str, List[str]]:
    """
    Create stratified train/validation/test splits.

    Maintains proportional representation across all accession count strata.

    Args:
        ground_truth_dict: Dictionary from load_all_ground_truth()
        split_ratios: Dict with 'development', 'validation', 'test' ratios
        random_seed: For reproducibility

    Returns:
        Dictionary with lists of PMCIDs for each split
    """
    if split_ratios is None:
        split_ratios = SPLIT_RATIOS

    #set random seed
    random.seed(random_seed)

    #Use default pinned documents if none provided
    if pinned_documents is None:
        pinned_documents = PINNED_DOCUMENTS

    #Step 1: Separate Pinned from unpinned docs

    pinned_dev = []
    pinned_val = []
    pinned_test = []
    unpinned_pmcids = []

    for pmcid in ground_truth_dict.keys():
        if pmcid in pinned_documents["development"]:
          pinned_dev.append(pmcid)
          print(f"pinned to development : {pmcid}")
        elif pmcid in pinned_documents["validation"]:
          pinned_val.append(pmcid)
          print(f"pinned to validation : {pmcid}")
        elif pmcid in pinned_documents["test"]:
          pinned_test.append(pmcid)
          print(f"pinned to test : {pmcid}")
        else:
          unpinned_pmcids.append(pmcid)

    #if pinned doc not in found in ground truth, provide warnning
    all_pinned = (pinned_documents["development"] +
                  pinned_documents["validation"] +
                  pinned_documents["test"])
    for pmcid in all_pinned:
      if pmcid not in ground_truth_dict.keys():
        print(f"Warning: {pmcid} not found in ground truth")


    #Step 2: Group unpinned documents by stratum
    strata_groups = {
        "zero": [],
        "low": [],
        "medium": [],
        "high": []
    }

    for pmcid in unpinned_pmcids:
       stratum = ground_truth_dict[pmcid]['stratum']
       strata_groups[stratum].append(pmcid)

    #Step 3 calculate adjusted ratios

    #account for pinned document when calc split sizes
    total_unpinned = len(unpinned_pmcids)
    total_all = len(ground_truth_dict)

    #tqrget sizes accounting for pinned docs
    target_dev = round(total_all * split_ratios["development"]) - len(pinned_dev)
    target_val = round(total_all * split_ratios["validation"]) - len(pinned_val)
    target_test = total_unpinned - target_dev - target_val

    #make sure non-negative targets
    target_dev  = max(0, target_dev)
    target_val = max(0, target_val)
    target_test = max(0, target_test)

    #Step 4 Stratified sampling of unpinned docs
    dev_set = list(pinned_dev)
    val_set = list(pinned_val)
    test_set = list(pinned_test)

    for stratum_name, pmcid_list in strata_groups.items():
        random.shuffle(pmcid_list)

        stratum_total = len(pmcid_list)
        if stratum_total == 0:
           continue

        #calc sizes for this stratum(from unpinned)
        stratum_dev = round(stratum_total *(target_dev/total_unpinned)) if total_unpinned > 0 else 0
        stratum_val = round(stratum_total * (target_val/total_unpinned)) if total_unpinned > 0 else 0

        #assign to sets
        dev_set.extend(pmcid_list[0:stratum_dev])
        val_set.extend(pmcid_list[stratum_dev:stratum_dev+stratum_val])
        test_set.extend(pmcid_list[stratum_dev + stratum_val:])


    splits = {
        "development": dev_set,
        "validation": val_set,
        "test": test_set
    }

    return splits

In [10]:
def print_split_summary(
    splits: Dict[str, List[str]],
    ground_truth_dict: Dict[str, dict]
) -> None:
    """
    Print summary of the stratified splits.
    """

    print("\n" + "=" * 70)
    print("STRATIFIED SPLIT SUMMARY")
    print("=" * 70)

    total = sum(len(s) for s in splits.values())

    for split_name, pmcids in splits.items():
        print(f"\n{split_name.upper()} SET: {len(pmcids)} documents ({len(pmcids)/total*100:.1f}%)")

        #count strate within this split
        strata_counts = Counter()

        for pmcid in pmcids:
            stratum = ground_truth_dict[pmcid]["stratum"]
            strata_counts[stratum] += 1

        print(f"  Stratum distribution:")
        for stratum_name in STRATA_BINS.keys():
            count = strata_counts.get(stratum_name, 0)
            pct = (count / len(pmcids) * 100) if len(pmcids) > 0 else 0
            print(f"    {stratum_name}: {count} ({pct:.1f}%)")

def save_split_assignments(
    splits: Dict[str, List[str]],
    ground_truth_dict: Dict[str, dict],
    output_folder: str,
    filename: str = "validation_splits.json",
    pinned_documents: Dict[str, List[str]] = None
) -> str:
    """
    Save split assignments to JSON file for reproducibility.

    Returns:
        Path to saved file
    """

    #use default pinned doc if none provided
    if pinned_documents is None:
        pinned_documents = PINNED_DOCUMENTS

    #Cehck output folder exists
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    output_path = Path(output_folder) / filename

    #Build o/p data with metadata
    output_data = {
        "metadata": {
            "random_seed": RANDOM_SEED,
            "split_ratios": SPLIT_RATIOS,
            "strata_bins": {k: list(v) for k,v in STRATA_BINS.items()},
            "total_documents": len(ground_truth_dict),
            "created_date": None
        },
        "splits": {},
        "document_details": {}
    }

    #Add creation date
    from datetime import datetime
    output_data["metadata"]["created_date"] = datetime.now().isoformat()

    #Add split assignments with pinned count

    for split_name, pmcids in splits.items():
        pinned_in_split = [p for p in pmcids if p in pinned_documents.get(split_name, [])]

        output_data["splits"][split_name] = {
            "count": len(pmcids),
            "pinned_count": len(pinned_in_split),
            "files": pmcids
        }

    #Add document details for reference
    for pmcid, doc_data in ground_truth_dict.items():
        output_data["document_details"][pmcid] ={
            "filename":doc_data["filename"],
            "accession_count": doc_data["accession_count"],
            "stratum": doc_data["stratum"],
            "accessions": doc_data["accessions"]
        }

    # Save
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)

    print(f"\nSplit assignments saved to: {output_path}")
    return str(output_path)



In [11]:
def main():
    """
    Main execution function - run all three tasks.
    """
    print("=" * 70)
    print("ACCESSION GROUND TRUTH ANALYSIS AND STRATIFIED SAMPLING")
    print("=" * 70)

    # Task A & B: Load and analyse ground truth
    print(f"\nLoading ground truth from: {GROUND_TRUTH}")
    ground_truth_dict = load_all_ground_truth(GROUND_TRUTH)

    if not ground_truth_dict:
        print("ERROR: No ground truth files loaded. Check the folder path.")
        return

    # Corrected logic to extract and print the list of PMCIDs #changed
    pmcid_list = list(ground_truth_dict.keys())
    print(f"The ground truth contains {len(pmcid_list)} PMCIDs.")

    # Use the recommended join method for cleaner output #changed
    print("Example PMCIDs (first 10) are:\n")
    print("\n".join(pmcid_list[:10]))


    #Analyse distribution
    statistics = analyse_ground_truth_distribution(ground_truth_dict)
    print_distribution_report(statistics)

    #Task C: Create stratified splits
    splits = create_stratified_three_set_split(ground_truth_dict)
    print_split_summary(splits, ground_truth_dict)

    save_split_assignments(splits, ground_truth_dict, OUTPUT_FOLDER)

    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETE")
    print("=" * 70)

    return ground_truth_dict, statistics, splits

In [12]:
if __name__ == "__main__":
    ground_truth_dict, statistics, splits = main()

ACCESSION GROUND TRUTH ANALYSIS AND STRATIFIED SAMPLING

Loading ground truth from: /content/drive/MyDrive/AI6129/ground_truth
Found 227 JSON files in folder
The ground truth contains 227 PMCIDs.
Example PMCIDs (first 10) are:

PMC4672624
PMC4748464
PMC4806280
PMC4810482
PMC4824889
PMC4859176
PMC4866840
PMC4881965
PMC4892500
PMC4932652

GROUND TRUTH ACCESSION DISTRIBUTION ANALYSIS

Total Documents: 227
Total Accessions: 1565
Mean Accessions per Document: 6.89
Min Accessions: 0
Max Accessions: 556
Documents with Zero Accessions: 116

----------------------------------------------------------------------
STRATIFICATION SUMMARY
----------------------------------------------------------------------
Stratum      Range        Count      Percentage  
----------------------------------------------------------------------
zero         0            116        51.1%
low          1-2          54         23.8%
medium       3-5          26         11.5%
high         6-+          31         13.7%

--

In [14]:
import json
from pathlib import Path

# ============================================================================
# CONFIGURATION
# ============================================================================

JSON_FOLDER = "/content/drive/MyDrive/AI6129/ground_truth"


# ============================================================================
# FUNCTIONS
# ============================================================================

def identify_outliers(folder_path: str):
    """
    Identify documents with zero accessions and the maximum accession document.
    """
    zero_accession_docs = []
    max_accession_doc = None
    max_count = 0

    json_files = list(Path(folder_path).glob("*.json"))

    for json_file in json_files:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Get PMCID
        pmcid = data.get("PMCID", json_file.stem)

        # Parse accessions (same logic as main script)
        accessions = []
        acc_list = data.get("merge_accession_number", [])
        if acc_list and isinstance(acc_list, list):
            accessions = [a for a in acc_list if a]

        if not accessions:
            acc_merge = data.get("merge_accession_number", "")
            if acc_merge and isinstance(acc_merge, str):
                accessions = [a.strip() for a in acc_merge.replace(",", " ").split() if a.strip()]

        # Strip [xxx] suffix from each accession  # changed
        accessions = [re.sub(r'\[.*?\]$', '', acc) for acc in accessions]

        # Remove duplicates
        accessions = list(dict.fromkeys(accessions))
        count = len(accessions)

        # Track zero accession documents
        if count == 0:
            zero_accession_docs.append({
                "pmcid": pmcid,
                "filename": json_file.name,
                "filepath": str(json_file),
                "title": data.get("title", "N/A"),
                #"abstract_snippet": data.get("Abstract", "")[:300]
            })

        # Track maximum
        if count > max_count:
            max_count = count
            max_accession_doc = {
                "pmcid": pmcid,
                "filename": json_file.name,
                "filepath": str(json_file),
                "count": count,
                "title": data.get("title", "N/A"),
                "accessions_sample": accessions[:20],  # First 20 for inspection
                "accessions_all": accessions
            }

    return zero_accession_docs, max_accession_doc


def print_outlier_report(zero_docs, max_doc):
    """
    Print formatted report of outliers.
    """
    print("=" * 70)
    print("OUTLIER DOCUMENT INSPECTION")
    print("=" * 70)

    # Zero accession documents
    print("\n" + "-" * 70)
    print("A) DOCUMENTS WITH ZERO ACCESSIONS")
    print("-" * 70)

    if not zero_docs:
        print("No documents with zero accessions found.")
    else:
        print(f"Found {len(zero_docs)} document(s):\n")
        for i, doc in enumerate(zero_docs, 1):
            print(f"{i}. PMCID: {doc['pmcid']}")
            print(f"   Filename: {doc['filename']}")
            print(f"   Title: {doc['title']}")
            #print(f"   Abstract: {doc['abstract_snippet']}...")
            print()

    # Maximum accession document
    # print("-" * 70)
    # print("B) DOCUMENT WITH MAXIMUM ACCESSIONS")
    # print("-" * 70)

    # if not max_doc:
    #     print("No documents found.")
    # else:
    #     print(f"\nPMCID: {max_doc['pmcid']}")
    #     print(f"Filename: {max_doc['filename']}")
    #     print(f"Title: {max_doc['title']}")
    #     print(f"Accession Count: {max_doc['count']}")
    #     print(f"\nFirst 20 accessions:")
    #     for i, acc in enumerate(max_doc['accessions_sample'], 1):
    #         print(f"  {i:2}. {acc}")

    #     if max_doc['count'] > 20:
    #         print(f"  ... and {max_doc['count'] - 20} more")


# ============================================================================
# MAIN
# ============================================================================

def main():
    # Mount Drive for Colab
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except ImportError:
        pass

    print(f"Scanning: {JSON_FOLDER}\n")

    zero_docs, max_doc = identify_outliers(JSON_FOLDER)
    print_outlier_report(zero_docs, max_doc)

    return zero_docs, max_doc


if __name__ == "__main__":
    zero_docs, max_doc = main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scanning: /content/drive/MyDrive/AI6129/ground_truth

OUTLIER DOCUMENT INSPECTION

----------------------------------------------------------------------
A) DOCUMENTS WITH ZERO ACCESSIONS
----------------------------------------------------------------------
Found 116 document(s):

1. PMCID: PMC4672624
   Filename: PMC4672624.json
   Title: Cardiac device infection with Salmonella Blockley.

2. PMCID: PMC4748464
   Filename: PMC4748464.json
   Title: A multi-drug resistant Salmonella Typhimurium ST213 human-invasive strain (33676) containing the blaCMY-2 gene on an IncF plasmid is attenuated for virulence in BALB/c mice.

3. PMCID: PMC4806280
   Filename: PMC4806280.json
   Title: Salmonella Typhi Vertebral Osteomyelitis and Epidural Abscess.

4. PMCID: PMC4939201
   Filename: PMC4939201.json
   Title: First Case of Lung Abscess due to Salmonella enterica Ser